# 🔗 Multivariate Prophet: Adding Regressors

## 1. What are Regressors?
In standard Prophet, the forecast is based purely on time:
$$y(t) = \text{Trend}(t) + \text{Seasonality}(t)$$

When you add a **Regressor** (also called an External Feature), you add a linear component based on another variable $X$:
$$y(t) = \text{Trend}(t) + \text{Seasonality}(t) + \beta \cdot X(t)$$

* **Example:** Predicting *Ice Cream Sales* ($y$).
* **Regressor:** *Temperature* ($X$).
* **Logic:** If Temperature goes up, Sales go up (Coefficient $\beta$ is positive).

## 2. The Golden Rule: "The Future Must Be Known" ⚠️
This is the most critical constraint of `add_regressor`.
**To predict $y$ for next month, you must know the value of $X$ for next month.**

* **Valid Use Case:**
    * **Holidays:** We know when Christmas is next year.
    * **Planned Promotions:** We know we will run a 20% discount next week.
    * **Controlled Variables:** We know our Ad Budget for Q4.
* **Invalid Use Case:**
    * **Stock Price:** You cannot use "Oil Price" to predict "Airline Stock" because you don't know the Oil Price next week! (Unless you forecast Oil Price first, which adds error).

## 3. How to use it
1.  Add the column to your training dataframe.
2.  Call `m.add_regressor('column_name')` **before** fitting.
3.  **Crucial:** When you create the `future` dataframe, you must populate this column with known future values.

#### 1) Importing Necessary Libraries:

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import statsmodels.api as sm

from prophet import Prophet
from prophet.utilities import regressor_coefficients

from sklearn.metrics import mean_absolute_percentage_error

#### 2) Loading Data:

In [2]:
# Using Dataset from Statsmodels:

macro_data = sm.datasets.macrodata.load_pandas().data

In [4]:
macro_data.head()

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint
0,1959.0,1.0,2710.349,1707.4,286.898,470.045,1886.9,28.98,139.7,2.82,5.8,177.146,0.00,0.00
1,1959.0,2.0,2778.801,1733.7,310.859,481.301,1919.7,29.15,141.7,3.08,5.1,177.830,2.34,0.74
2,1959.0,3.0,2775.488,1751.8,289.226,491.260,1916.4,29.35,140.5,3.82,5.3,178.657,2.74,1.09
3,1959.0,4.0,2785.204,1753.7,299.356,484.052,1931.3,29.37,140.0,4.33,5.6,179.386,0.27,4.06
4,1960.0,1.0,2847.699,1770.5,331.722,462.199,1955.5,29.54,139.6,3.50,5.2,180.007,2.31,1.19


In [5]:
macro_data.shape

(203, 14)

In [6]:
macro_data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 203 entries, 0 to 202
Data columns (total 14 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   year      203 non-null    float64
 1   quarter   203 non-null    float64
 2   realgdp   203 non-null    float64
 3   realcons  203 non-null    float64
 4   realinv   203 non-null    float64
 5   realgovt  203 non-null    float64
 6   realdpi   203 non-null    float64
 7   cpi       203 non-null    float64
 8   m1        203 non-null    float64
 9   tbilrate  203 non-null    float64
 10  unemp     203 non-null    float64
 11  pop       203 non-null    float64
 12  infl      203 non-null    float64
 13  realint   203 non-null    float64
dtypes: float64(14)
memory usage: 22.3 KB


In [8]:
macro_data.describe().T

,count,mean,std,min,25%,50%,75%,max
year,203.0,1983.876847,14.686817,1959.000,1971.0000,1984.000,1996.5000,2009.000
quarter,203.0,2.492611,1.118563,1.000,1.5000,2.000,3.0000,4.000
realgdp,203.0,7221.171901,3214.956044,2710.349,4440.1035,6559.594,9629.3465,13415.266
realcons,203.0,4825.293103,2313.346192,1707.400,2874.1000,4299.900,6398.1500,9363.600
realinv,203.0,1012.863862,585.102267,259.764,519.1475,896.210,1436.6815,2264.721
realgovt,203.0,663.328640,140.863655,460.400,527.9595,662.412,773.0490,1044.088
realdpi,203.0,5310.540887,2423.515977,1886.900,3276.9500,4959.400,6977.8500,10077.500
cpi,203.0,105.075788,61.278878,28.980,41.0500,104.100,159.6500,218.610
m1,203.0,667.927586,455.346381,139.600,228.6500,540.900,1102.1000,1673.900
tbilrate,203.0,5.311773,2.803071,0.120,3.5150,5.010,6.6650,15.330


##### Creating Date column from year and quarter:

In [10]:
macro_data['ds'] = pd.PeriodIndex.from_fields(
    year = macro_data['year'],
    quarter = macro_data['quarter'],
    freq= 'Q'
).to_timestamp()

In [11]:
macro_data.head()

,year,quarter,realgdp,realcons,realinv,realgovt,realdpi,cpi,m1,tbilrate,unemp,pop,infl,realint,ds
0,1959.0,1.0,2710.349,1707.4,286.898,470.045,1886.9,28.98,139.7,2.82,5.8,177.146,0.00,0.00,1959-01-01
1,1959.0,2.0,2778.801,1733.7,310.859,481.301,1919.7,29.15,141.7,3.08,5.1,177.830,2.34,0.74,1959-04-01
2,1959.0,3.0,2775.488,1751.8,289.226,491.260,1916.4,29.35,140.5,3.82,5.3,178.657,2.74,1.09,1959-07-01
3,1959.0,4.0,2785.204,1753.7,299.356,484.052,1931.3,29.37,140.0,4.33,5.6,179.386,0.27,4.06,1959-10-01
4,1960.0,1.0,2847.699,1770.5,331.722,462.199,1955.5,29.54,139.6,3.50,5.2,180.007,2.31,1.19,1960-01-01


##### Creating Dataframe of Date, Regressor and Target Variable:
(We will predict Consumption (y) based on Income (realdpi))